# Solution C1 · Breaking the tree with a trend: flat predictions outside the training range

**Type**: Break the model

## Objective

Build a series with trend and seasonality, train a linear regression and a regression tree on the first 200 months and predict the next 100: the tree is wrong by almost five times as much as the line, because outside the range of `t` it repeats the last value. Then fix it with a **hybrid** model
(linear trend + tree on the residuals).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor


def series():
    rng = np.random.default_rng(0)
    t = np.arange(300)
    y = 2 + 0.05 * t + 3 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.7, 300)
    return t, np.c_[t, np.sin(2 * np.pi * t / 12), np.cos(2 * np.pi * t / 12)], y


t, F, y = series()
tr, te = t < 200, t >= 200
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))

## Commented solution

In [ ]:
lin = LinearRegression().fit(F[tr], y[tr])
tree = DecisionTreeRegressor(min_samples_leaf=5, random_state=0).fit(F[tr], y[tr])
train_rmse = {"linear": rmse(lin.predict(F[tr]), y[tr]), "tree": rmse(tree.predict(F[tr]), y[tr])}
future_rmse = {"linear": rmse(lin.predict(F[te]), y[te]), "tree": rmse(tree.predict(F[te]), y[te])}
print("RMSE on the training set:", {k: round(v, 2) for k, v in train_rmse.items()}, "| in the future:", {k: round(v, 2) for k, v in future_rmse.items()})

In [ ]:
t_only = DecisionTreeRegressor(min_samples_leaf=5, random_state=0).fit(t[tr].reshape(-1, 1), y[tr])
future_tree_predictions = t_only.predict(t[te].reshape(-1, 1))
n_distinct_values = int(np.unique(future_tree_predictions).size)
last_constant_value = float(future_tree_predictions[-1])
print(f"distinct values predicted by the tree on t for the 100 future months: {n_distinct_values} (value {last_constant_value:.2f}) | mean of the last 30 training months: {y[tr][-30:].mean():.2f}")

In [ ]:
residuals = y[tr] - lin.predict(F[tr])
res_tree = DecisionTreeRegressor(min_samples_leaf=5, random_state=0).fit(F[tr], residuals)
hybrid_rmse = rmse(lin.predict(F[te]) + res_tree.predict(F[te]), y[te])
error_ratio = future_rmse["tree"] / future_rmse["linear"]
print(f"RMSE in the future: line {future_rmse['linear']:.2f} | tree {future_rmse['tree']:.2f} ({error_ratio:.1f} times) | hybrid {hybrid_rmse:.2f}")

## Check

In [ ]:
with verify():
    check(set(train_rmse) == {"linear", "tree"} and abs(train_rmse["tree"] - train_rmse["linear"]) < 0.1, "on the training set the two models are on par (within 0.1): nothing hints at the problem", "compare the training errors")
    check(future_rmse["linear"] < 1.0 and future_rmse["tree"] > 3, "in the future the line stays below 1.0; the tree above 3", "test error (t >= 200)")
    check(n_distinct_values is not None and n_distinct_values <= 3 and abs(last_constant_value - y[tr][-30:].mean()) < 3, "a tree on t predicts very few distinct values in the future (at most 3), close to the level of the last months", "the future falls entirely in the last leaf")
    check(hybrid_rmse is not None and hybrid_rmse < 1.0 and hybrid_rmse < future_rmse["tree"] / 3, "the hybrid in the future has RMSE below 1.0 (over 3 times better than the tree)", "line + tree on the residuals")
    check(error_ratio > 4, "the error of the tree is over 4 times that of the line", "future_rmse['tree'] / future_rmse['linear']")
done()

## What you learned

- **A tree does not extrapolate:** on the training set it is on par with the line (RMSE 0.67 for both: no hint of the problem), but in the future its error is almost five times larger (3.7 against 0.78). With `t` alone as a feature it predicts **a single value** (the mean of the last leaf) for all the 100 future months.
- The same holds for **random forest and gradient boosting**: they are sums and averages of trees, so they are constant outside the range seen. With data that have a trend (sales, prices, sensors with drift) they are not reliable "as they are".
- **The remedy is to model the trend separately:** a linear trend (or a differencing) and a tree on the **residuals** for the nonlinear part (seasonality, interactions). The hybrid has error 0.74 in the future: almost that of the line, with the flexibility of the tree on the stationary part.
- Alternatives: transform the target (differences, ratios with respect to a previous period) so that it is stationary, or use features that carry the trend (`t` as a feature is not enough for a tree: it is the target transformations that help it).

## Common pitfall

Validating a tree model with a random K-fold on data with a trend: every validation point has neighbors in training on both sides and the tree interpolates well. The flaw is discovered only in production, in the future (M3, C1: the splitter must mirror the use).

## Interview question

<details><summary>Why is a random forest not suited to predicting a series with a trend?</summary>

Because trees predict means of values seen in training: outside the range of the data the prediction is flat, so it cannot continue a trend. You remove the trend (linear regression, differencing), model the rest with the tree and put it back together; or you use time series models.
And you validate with a temporal split, which shows the problem right away.

</details>